In [ ]:
import glob
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import marsilea as ma
import marsilea.plotter as mp
import scienceplots
# %matplotlib widget
import pickle
plt.style.use(['science', 'nature'])
import numpy as np
from itertools import combinations, product
import pingouin as pg
from matplotlib.patches import Rectangle
from marsilea.plotter import FixedChunk
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

import marsilea as ma
import marsilea.plotter as mp

plt.rcParams['xtick.labelsize'] = 5
plt.rcParams['ytick.labelsize'] = 5
plt.rcParams['axes.labelsize'] = 6
plt.rcParams["xtick.top"] = False
plt.rcParams["ytick.right"] = False
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["legend.fontsize"] = 6
plt.rcParams['hatch.linewidth'] = 0.5

# %matplotlib widget
tool_map = {
    "scapa": "scAPA",
    "scapatrap": "scAPAtrap",
    "sierra": "Sierra",
    "maaper": "MAAPER",
    "scapture": "SCAPTURE",
    "scape": "SCAPE",
    "infernape": "Infernape",
}

protocol_map = {
    "Visium": "10X Visium",
    "VisiumHD": "10X Visium HD",
    "Chromium": "10X Chromium",
    "Dropseq": "Drop-seq",
    "Stereoseq": "Stereo-seq",
    "Slideseq": "Slide-seq V2",
    "SpatialTranscriptomics": "ST",
    "Microwell": "Microwell-seq",
}

protocol_order = ["10X Chromium", "Drop-seq", "Microwell-seq", "10X Visium","Stereo-seq", "Slide-seq V2", "ST"]
tool_order = ["scAPAtrap", "SCAPE", "Infernape", "SCAPTURE", "scAPA",  "Sierra"]

# order = ["10X Chromium", "Drop-seq", "Microwell-seq", "10X Visium", "10X Visium HD","Stereo-seq", "Slide-seq V2", "Spatial Transcriptomics"]

color = [
    "#386b98",
    "#269a51",
    "#edaa4d",
    "#d34123",
    "#7e648a",
    "#454545",
    "#929292",
]
palette=sns.color_palette(color, 7)
mm = 1/25.4

In [ ]:
palette

In [ ]:
# de_apa_performance_df.to_feather('/path/to/apabenchmark_final/data/performance/de_apa_performance.feather')
de_apa_performance_df = pd.read_feather('../../data/result/merged_tool_performance/de_apa_performance.feather')

In [ ]:
de_apa_performance_df['tool'] = de_apa_performance_df['tool'].map(tool_map)
de_apa_performance_df['protocol'] = de_apa_performance_df['protocol'].map(protocol_map)

In [ ]:
de_apa_performance_subset_df = de_apa_performance_df[de_apa_performance_df["filter_type_1"].str.endswith("0.05")]

In [ ]:
de_apa_performance_subset_df["gt_recall"] = de_apa_performance_subset_df["gt_count"] / 5000

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
# 按 'sample' 和 'tool' 分组
grouped = de_apa_performance_subset_df.groupby(['sample', 'tool',"match_type"])

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
from tqdm import tqdm


# 1. 创建一个空列表，用于存储处理后的每个分组
processed_groups_list = []

# 2. 按 'sample', 'tool', 和 'match_type' 分组
# 3. 使用 for 循环遍历每个分组
# group_keys 是一个包含分组键的元组, e.g., ('Stereoseq_mouse_embryo_CNS', 'SCAPE', 'gt_100')
# group_df 是对应分组的子 DataFrame
for group_keys, group_df in tqdm(grouped):
    
    # 为了避免 SettingWithCopyWarning，我们显式地创建一个副本
    current_group = group_df.copy()
    
    # 准备自变量
    X = current_group[['gt_recall']]

    # 检查数据点是否足够进行回归
    if len(current_group) < 2:
        # 如果不足，则用 NaN 填充结果列
        for col in ['f1', 'rank']:
            current_group[f'{col}_intercept'] = np.nan
            current_group[f'{col}_coefficient'] = np.nan
            current_group[f'{col}_r2'] = np.nan
            current_group[f'{col}_residuals'] = np.nan
    else:
        # --- 第一次回归: f1 vs gt_recall ---
        y_f1 = current_group['f1']
        model_f1 = LinearRegression()
        model_f1.fit(X, y_f1)
        
        # 将 f1 回归结果添加到当前分组的DataFrame中
        current_group['f1_intercept'] = model_f1.intercept_
        current_group['f1_coefficient'] = model_f1.coef_[0]
        current_group['f1_r2'] = model_f1.score(X, y_f1)
        current_group['f1_residuals'] = y_f1 - model_f1.predict(X)

        # # --- 第二次回归: rank vs gt_recall ---
        # if 'rank' in current_group.columns:
        #     y_rank = current_group['rank']
        #     model_rank = LinearRegression()
        #     model_rank.fit(X, y_rank)

        #     # 将 rank 回归结果添加到当前分组的DataFrame中
        #     current_group['rank_intercept'] = model_rank.intercept_
        #     current_group['rank_coefficient'] = model_rank.coef_[0]
        #     current_group['rank_r2'] = model_rank.score(X, y_rank)
        #     current_group['rank_residuals'] = y_rank - model_rank.predict(X)
        # else:
        #     # 如果没有rank列，用NaN填充
        #     current_group['rank_intercept'] = np.nan
        #     current_group['rank_coefficient'] = np.nan
        #     current_group['rank_r2'] = np.nan
        #     current_group['rank_residuals'] = np.nan
            
    # --- 这里是查看中间文件的绝佳位置 ---
    # print(f"\n--- Processing Group: {group_keys} ---")
    # print("Data for this group with regression results added:")
    # print(current_group)
    # print("--------------------------------------")
    
    # 4. 将处理完毕的当前分组添加到列表中
    processed_groups_list.append(current_group)

# 5. 循环结束后，将列表中的所有DataFrame合并成一个
if processed_groups_list:
    final_df = pd.concat(processed_groups_list)
else:
    final_df = df.copy() # 如果原始df为空或没有有效分组，则返回原始df的副本

In [ ]:
final_df["mean_f1"] = final_df.groupby(["tool", "sample", "match_type"])["f1"].transform('mean')

In [ ]:
filter_type_comb_set = [
    "wilcox_DWUI_0.05|dexseq_log2fc_0.5",
    "wilcox_DWUI_0.05|dexseq_log2fc_1",
    "wilcox_DWUI_0.05|dexseq_log2fc_1.25",
]

In [ ]:
final_df["filter_type_comb"] = final_df["filter_type_1"] + "|" + final_df["filter_type_2"]

In [ ]:
de_apa_performance_subset_df["filter_type_comb"] = de_apa_performance_subset_df["filter_type_1"] + "|" + de_apa_performance_subset_df["filter_type_2"]
final_set_df = de_apa_performance_subset_df[de_apa_performance_subset_df["filter_type_comb"].isin(filter_type_comb_set)].groupby(["tool", "sample", "match_type"]).mean().reset_index()

In [ ]:
final_set_df = final_df[final_df["filter_type_comb"].isin(filter_type_comb_set)].groupby(["tool", "sample", "match_type"]).mean().reset_index()

In [ ]:
residual_df = final_df.groupby(["filter_type_comb"])[["rank_residuals","f1_residuals","gt_recall"]].mean().reset_index()
residual_df["filter_type_1"] = residual_df["filter_type_comb"].apply(lambda x: x.split('|')[0])
residual_df["filter_type_2"] = residual_df["filter_type_comb"].apply(lambda x: x.split('|')[1])

In [ ]:
filter_type_1_map = {
    "dexseq_0.05": "DexSeq",
    "fisher_0.05": "Fisher",
    "wilcox_PDUI_0.05": "Wilcox(PDUI)",
    "wilcox_PPUI_0.05": "Wilcox(PPUI)",
    "wilcox_DWUI_0.05": "Wilcox(DWUI)",
    "wilcox_RWUI_0.05": "Wilcox(RWUI)",
}

def filter_type_2_map_func(x):
    if x.startswith("dexseq_log2fc"):
        return x.replace("dexseq_log2fc_", "DexSeq log2FC ")
    elif x.startswith("MPRO"):
        return x.replace("_", "\\textgreater")
    else:
        return x.replace("_", "diff \\textgreater")

filter_type_1_order = [
    "dexseq_0.05",
    "fisher_0.05",
    "wilcox_PDUI_0.05",
    "wilcox_PPUI_0.05",
    "wilcox_RWUI_0.05",
    "wilcox_DWUI_0.05",
]

filter_type_2_order = [
    "PDUI_0.1",
    "PDUI_0.2",
    "PDUI_0.3",
    "PDUI_0.4",
    "PDUI_0.5",
    "PPUI_0.1",
    "PPUI_0.2",
    "PPUI_0.3",
    "PPUI_0.4",
    "PPUI_0.5",
    "RWUI_0.1",
    "RWUI_0.2",
    "RWUI_0.3",
    "RWUI_0.4",
    "RWUI_0.5",
    "DWUI_0.1",
    "DWUI_0.2",
    "DWUI_0.3",
    "DWUI_0.4",
    "DWUI_0.5",
    "MPRO_0.1",
    "MPRO_0.2",
    "MPRO_0.3",
    "MPRO_0.4",
    "MPRO_0.5",
    "dexseq_log2fc_0.25",
    "dexseq_log2fc_0.5",
    "dexseq_log2fc_0.75",
    "dexseq_log2fc_1",
    "dexseq_log2fc_1.25",
]

filter_type_1_label = [
    "DexSeq",
    "Fisher",
    "Wilcox PDUI",
    "Wilcox PPUI",
    "Wilcox RWUI",
    "Wilcox DWUI"
]

In [ ]:
residual_df["filter_type_1"] = pd.Categorical(residual_df["filter_type_1"], categories=filter_type_1_order)
residual_df["filter_type_2"] = pd.Categorical(residual_df["filter_type_2"], categories=filter_type_2_order)
residual_df = residual_df.sort_values(["filter_type_1", "filter_type_2"])

In [ ]:
plt.rcParams["xtick.direction"] = "out"
plt.rcParams["ytick.direction"] = "out"
plt.rcParams["xtick.bottom"] = False
plt.rcParams["xtick.minor.bottom"] = True
plt.rcParams["ytick.left"] = False

cmap = ["Reds", "Blues", "Greens"]
h = ma.Heatmap(
    residual_df[["filter_type_1", "filter_type_2", "f1_residuals"]].pivot(index="filter_type_1", columns="filter_type_2"),
    cmap="RdBu",
    label="Mean Residual",
    width=80*mm,
    height=40*mm,
    cbar_kws={"width":1, "height":10}, #"orientation": "horizontal",
)
# h.add_layer(mp.TextMesh(np.around(de_performance_max_f1_df.to_numpy(),2), color="black", fontsize=5))
h.add_right(mp.Numbers(
    residual_df[["filter_type_1", "filter_type_2", "f1_residuals"]].pivot(index="filter_type_1", columns="filter_type_2").mean(axis=1),
    color=palette[0],
    label="Mean Residual",
    show_value=False,
    props={"fontsize": 5}
    ),
    size=8*mm,
    pad=0.5*mm
)
h.add_top(mp.Numbers(
    residual_df[["filter_type_1", "filter_type_2", "f1_residuals"]].pivot(index="filter_type_1", columns="filter_type_2").mean(),
    color=palette[0], 
    label="Mean DE APA F1",
    show_value=False,
    props={"fontsize": 5}
    ),
    size=10*mm,
    pad=1*mm,
    name="top"
)

var_num=5
h.cut_cols(list(range(var_num, len(filter_type_2_order), var_num)), spacing=0.01)

# h.add_left(mp.Chunk(["" for i in range(7)], palette, ),pad=0.05)
h.add_left(mp.Labels(filter_type_1_label, rotation=0, fontsize=6), size=7*mm, pad=0.5*mm)
# # h.add_bottom(mp.Chunk(["" for i in range(7)], palette),pad=0.05)
h.add_bottom(mp.Labels([filter_type_2_map_func(i) for i in filter_type_2_order], rotation=90, fontsize=6), size=10*mm, pad=0.5*mm)
h.add_legends("right")
h.render()

# h.get_ax("top").yaxis.set_label_position("right")
# h.get_ax("top").yaxis.set_ticks_position("right")
# h.get_ax("top").tick_params(axis='y', which='minor', right=False)
# h.get_ax("top").spines["right"].set_visible(True)
# h.get_ax("top").spines["left"].set_visible(False)

plt.savefig('../../figures/filter_residuals_heatmap.pdf', bbox_inches='tight', dpi=300)
# plt.rcParams["xtick.direction"] = "in"
# plt.rcParams["ytick.direction"] = "in"
# plt.rcParams["xtick.bottom"] = True
# plt.rcParams["ytick.left"] = True
# plt.rcParams["xtick.minor.bottom"] = False

In [ ]:
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.bottom"] = True
plt.rcParams["ytick.left"] = True
plt.rcParams["xtick.minor.bottom"] = False

In [ ]:
def format_filter_type_comb(text):
    """
    格式化 filter_type_comb 字符串。

    规则:
    1. 去掉'|'前的'_0.05'。
    2. 如果'|'后只有一个'_'，换成'>'。
    3. 如果'|'后有两个'_'，第一个换成' '，第二个换成'>'。
    """
    if '|' not in text:
        return text

    # 将字符串按 '|' 分割
    prefix, suffix = text.split('|', 1)

    # 1. 处理前缀
    formatted_prefix = prefix.replace('_0.05', '')
    if formatted_prefix.startswith('wilcox'):
        formatted_prefix = "Wilcox (" + formatted_prefix.replace('wilcox_', '').upper() + ")"

    # 2. 处理后缀
    if suffix.count('_') == 2:
        # 先替换第一个'_'为空格，再替换剩下的'_'为'>'
        formatted_suffix = suffix.replace('_', ' ', 1).replace('_', '\\textgreater').capitalize()
    elif suffix.count('_') == 1:
        # 替换唯一的'_'为'>'
        formatted_suffix = (suffix.upper()).replace('_', ' diff\\textgreater')
    else:
        # 如果没有或超过两个下划线，保持原样
        formatted_suffix = suffix

    # 重新组合字符串
    return f"{formatted_prefix}+{formatted_suffix}"

In [ ]:
# 假设 mm 用于将毫米转换为英寸
mm = 1/25.4 

plt.figure(figsize=(45*mm, 70*mm))

# 1. 绘制散点图
sns.scatterplot(data=residual_df, x="f1_residuals", y="gt_recall",
                color='grey', alpha=0.5, label='_nolegend_')

# 2. 在Y轴上进行划分并绘制水平虚线
y_min = residual_df['gt_recall'].min()
y_max = residual_df['gt_recall'].max()
third_1 = y_min + (y_max - y_min) / 3
third_2 = y_min + 2 * (y_max - y_min) / 3
plt.axhline(y=third_1, color='black', linestyle='--', linewidth=0.8)
plt.axhline(y=third_2, color='black', linestyle='--', linewidth=0.8)

# 3. 根据Y轴的值划分数据
section_1 = residual_df[residual_df['gt_recall'] <= third_1]
section_2 = residual_df[(residual_df['gt_recall'] > third_1) & (residual_df['gt_recall'] <= third_2)]
section_3 = residual_df[residual_df['gt_recall'] > third_2]

# --- 标注逻辑 ---
point_colors = [
    "#386b98", "#269a51", "#edaa4d", "#d34123",
    "#7e648a", "#454545", "#929292",
]
color_index = 0
sections = [section_1, section_2, section_3]
for i, section in enumerate(sections):
    if not section.empty:
        top_two_points = section.nlargest(2, 'f1_residuals')
        top_two_points = top_two_points.sort_values(by='f1_residuals', ascending=True)
        max_point = top_two_points.iloc[0]
        x_coord = max_point['f1_residuals']
        y_coord = max_point['gt_recall']
        formatted_label = format_filter_type_comb(max_point['filter_type_comb'])
        current_color = point_colors[color_index]
        
        plt.scatter(x_coord, y_coord, color=current_color, 
                    label=formatted_label, s=10, zorder=5)
        
        plt.annotate(
            text='',
            xy=(x_coord, y_coord),
            xytext=(x_coord - 0.005, y_coord + 0.01),
            arrowprops=dict(arrowstyle="->", color=current_color, linewidth=1.5)
        )
        color_index += 1

# --- 新增和修改的部分 ---

# 4. 将X轴移动到顶部
ax = plt.gca()  # 获取当前坐标轴
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

# 5. 设置坐标轴标签
plt.xlabel('Mean Residual')
plt.ylabel('Ground Truth Recall')

# 6. 将图例移动到底部，并设置为水平排列
#    - loc='lower center' 表示图例的锚点在图例自身的“下边中间”
#    - bbox_to_anchor=(0.5, -0.4) 表示将锚点放在画布坐标的 (50%, -40%) 位置
#    - ncols=3 表示图例有3列
plt.legend(bbox_to_anchor=(0.5, -0.25), loc='lower center', 
           borderaxespad=0., fontsize=6, ncols=1)
plt.savefig('../../figures/max_residuals_vs_gt_recall_with_arrows.pdf', dpi=300)

# 调整布局以确保移动到底部的图例不会被裁切
# plt.tight_layout(rect=[0, 0, 1, 1.1]) # 给顶部留出空间给移动后的X轴

# 显示图表
plt.show()

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns


mm = 1/25.4

plt.rcParams['hatch.color'] = 'black'
tool_order = ["scAPAtrap", "SCAPE", "Infernape", "SCAPTURE", "scAPA", "Sierra"]
hue_order = ['gt_25', 'gt_50', 'gt_75', 'gt_100', 'pd_25', 'pd_50', 'pd_75', 'pd_100', 'te']
metrics = ['precision', 'recall', 'f1']
pd_indices = [4, 5, 6, 7]

# 定义基础颜色
color = [
    "#386b98",
    "#269a51",
    "#edaa4d",
    "#d34123",
    "#7e648a",
    "#454545",
    "#929292",
]

# --- 主要修改部分 ---

# 1. 根据您的逻辑创建自定义调色板
# gt_25, gt_50, gt_75, gt_100 使用前4色
# pd_25, pd_50, pd_75, pd_100 也使用前4色
# te 使用第5色
custom_palette = [
    color[0], color[1], color[2], color[3],  # for gt_*
    color[0], color[1], color[2], color[3],  # for pd_*
    color[4]                                                  # for te
]

# 定义条纹样式
hatch_pattern = '///////'

# 设置绘图样式
# sns.set_theme(style="whitegrid", font_scale=1.2)
plt.figure(figsize=(185*mm, 100*mm))

outlier_params = {
    'flierprops': {
        'marker': 'o',
        'markersize': 0.1,  # 调整点大小
        'markerfacecolor': 'gray',
        'markeredgecolor': 'gray'
    }
}

# 遍历三个评估指标
metrics = ['precision', 'recall', 'f1']
for i, metric in enumerate(metrics, 1):
    plt.subplot(3, 1, i)
    
    # 绘制箱线图
    ax = sns.boxplot(
        data=final_set_df,
        x="tool",
        y=metric,
        hue="match_type",
        hue_order=hue_order,
        order=tool_order,
        palette=custom_palette,
        dodge=True,
        width=0.8,
        **outlier_params
    )
    ax.grid(linestyle='--', alpha=0.6, linewidth=0.5)
    # 设置坐标轴和标签
    # ax.set_title(metric.capitalize(), fontsize=6, weight='bold')
    ax.set_xlabel("", fontsize=6)
    ax.set_ylabel(metric.capitalize(), fontsize=6)
    # ax.tick_params(axis='x', rotation=45)

    # --- 关键修复：直接访问箱线图的Patch对象 ---
    # 获取所有箱线图的矩形元素
    n_groups = len(tool_order)  # 工具数量
    n_hues = len(hue_order)     # 匹配类型数量
    
    # 找到所有箱体的矩形元素（每个箱体有多个patch元素）
    box_patches_with_x = []
    for child in ax.get_children():
        if isinstance(child, mpl.patches.PathPatch):
            # 检查是否是箱体本身（而不是中位数线等）
            if len(child.get_path().vertices) >= 5:  # 箱体多边形
                # 获取所有X坐标
                x_coords = child.get_path().vertices[:, 0]
                # 使用最小的X坐标作为箱体的排序依据
                min_x = np.min(x_coords)
                box_patches_with_x.append((min_x, child))

    # 根据X坐标对箱体进行排序
    box_patches_with_x.sort(key=lambda item: item[0])

    # 提取排序后的patch元素列表
    sorted_box_patches = [patch for x, patch in box_patches_with_x]
    
    for idx, patch in enumerate(sorted_box_patches):
        hue_idx = idx % n_hues
        tool_idx = idx // n_hues
        
        if hue_idx in pd_indices:
            patch.set_hatch(hatch_pattern)
            patch.set_edgecolor('k')
            patch.set_linewidth(0.6)

    if i == 1:
        # **修改**: 不再修改旧句柄，而是创建全新的图例句柄
        _, labels = ax.get_legend_handles_labels() # 只需要原始的标签
        
        new_handles = []
        for handle_idx, label in enumerate(labels):
            color = custom_palette[handle_idx]
            # 创建一个带黑色边框的色块
            patch = Patch(facecolor=color, edgecolor='black', label=label)
            # 如果是 'pd_' 类别，就给这个色块加上条纹
            if handle_idx in pd_indices:
                patch.set_hatch(hatch_pattern)
            new_handles.append(patch)

        # 将图例定位在整张图片右侧
        ax.legend(
            new_handles,
            labels,
            title="Match Type",
            title_fontsize=6,
            fontsize=6,
            frameon=False,
            bbox_to_anchor=(1.02, 0.5),  # 向右移动系数
            loc='center left',
            borderaxespad=0.
        )
    else:
        ax.get_legend().remove()

    if i == 3:
        ax.set_xlabel("Tool", fontsize=6)
    else:
        ax.set_xlabel("")
        ax.set_xticklabels([])
plt.savefig('../../figures/apa_detect_performance.pdf', bbox_inches='tight', dpi=300)


In [ ]:
final_set_df['protocol'] = final_set_df['sample'].str.split('_',expand=True)[0]
final_set_df['protocol'] = final_set_df['protocol'].map(protocol_map)

In [ ]:


# 数据预处理
# match_performance_df['tool'] = match_performance_df['tool'].map(tool_map)
# match_performance_df['protocol'] = match_performance_df['protocol'].map(protocol_map)

# 全局样式设定
# plt.rcParams.update({'font.size': 7})
PALETTE = palette
flierprops = dict(marker='o', markersize=0.5, markerfacecolor='gray', markeredgecolor='gray')

hue_order = ['gt_25', 'gt_50', 'gt_75', 'gt_100', 'pd_25', 'pd_50', 'pd_75', 'pd_100', 'te']
metrics = ['precision', 'recall', 'f1']
pd_indices = [4, 5, 6, 7]

custom_palette = [
    color[0], color[1], color[2], color[3],  # for gt_*
    color[0], color[1], color[2], color[3],  # for pd_*
    color[4]                                                  # for te
]

# 定义条纹样式
hatch_pattern = '///////'

# 遍历每个指标绘图
for metric in ['precision', 'recall', 'f1']:
    # 创建新画布，设定成A4纵向纸张级别的高度
    fig, axes = plt.subplots(
        nrows=len(protocol_order), 
        ncols=1,
        figsize=(60*mm, 170*mm),  # 宽180mm, 高280mm转英寸
        dpi=300
    )
    
    # 主标题（可选）
    # fig.suptitle(f"{metric.capitalize()} Across Protocols", y=0.98, fontsize=9)
    
    # 遍历每个protocol画子图
    for idx, (protocol, ax) in enumerate(zip(protocol_order, axes)):
        # 数据筛选
        plot_data = final_set_df[
            (final_set_df['protocol'] == protocol) &
            (final_set_df[metric].notna())
        ]
        
        # 绘制箱线图核心
        sns.boxplot(
            data=plot_data,
            x="tool",
            y=metric,
            hue="match_type",
            order=tool_order,
            hue_order=hue_order,
            palette=custom_palette,
            ax=ax,
            dodge=True,
            width=0.7,
            flierprops=flierprops,
            linewidth=0.4
        )
        
        # 子图装饰
        ax.set_title(f"{protocol}", loc='left', pad=2, fontsize=6)
        ax.set_xlabel("")  # 隐藏所有x轴标签
        ax.set_ylabel(metric.capitalize() if idx == 0 else "", labelpad=2)  # 仅第一列显示y标签
        ax.tick_params(axis='both', labelsize=6, pad=1)
        ax.grid(True, linestyle=':', linewidth=0.3)
        
            # --- 关键修复：直接访问箱线图的Patch对象 ---
        # 获取所有箱线图的矩形元素
        n_groups = len(tool_order)  # 工具数量
        n_hues = len(hue_order)     # 匹配类型数量
        
        # 找到所有箱体的矩形元素（每个箱体有多个patch元素）
        box_patches_with_x = []
        for child in ax.get_children():
            if isinstance(child, mpl.patches.PathPatch):
                # 检查是否是箱体本身（而不是中位数线等）
                if len(child.get_path().vertices) >= 5:  # 箱体多边形
                    # 获取所有X坐标
                    x_coords = child.get_path().vertices[:, 0]
                    # 使用最小的X坐标作为箱体的排序依据
                    min_x = np.min(x_coords)
                    box_patches_with_x.append((min_x, child))

        # 根据X坐标对箱体进行排序
        box_patches_with_x.sort(key=lambda item: item[0])

        # 提取排序后的patch元素列表
        sorted_box_patches = [patch for x, patch in box_patches_with_x]
        
        for i, patch in enumerate(sorted_box_patches):
            hue_idx = i % n_hues
            tool_idx = i // n_hues
            
            if hue_idx in pd_indices:
                patch.set_hatch(hatch_pattern)
                patch.set_edgecolor('k')
                patch.set_linewidth(0.6)

        if i == 1:
            # **修改**: 不再修改旧句柄，而是创建全新的图例句柄
            _, labels = ax.get_legend_handles_labels() # 只需要原始的标签
            
            new_handles = []
            for handle_idx, label in enumerate(labels):
                color = custom_palette[handle_idx]
                # 创建一个带黑色边框的色块
                patch = Patch(facecolor=color, edgecolor='black', label=label)
                # 如果是 'pd_' 类别，就给这个色块加上条纹
                if handle_idx in pd_indices:
                    patch.set_hatch(hatch_pattern)
                new_handles.append(patch)

            # 将图例定位在整张图片右侧
            ax.legend(
                new_handles,
                labels,
                title="Match Type",
                title_fontsize=6,
                fontsize=6,
                frameon=False,
                bbox_to_anchor=(1.02, 0.5),  # 向右移动系数
                loc='center left',
                borderaxespad=0.
            )
        else:
            ax.get_legend().remove()


        # 隐藏除最后一行外的x轴刻度标签
        if idx != len(protocol_order)-1:
            ax.set_xticklabels([])
        else:
            plt.xticks(fontsize=6, rotation=90)
        
        # 移除单个子图图例
        leg = ax.get_legend()
        if leg:
            leg.remove()

    # 全图统一定制图例
    # handles, labels = axes[0].get_legend_handles_labels()

    _, labels = axes[0].get_legend_handles_labels() # 只需要原始的标签
    
    new_handles = []
    for handle_idx, label in enumerate(labels):
        color = custom_palette[handle_idx]
        # 创建一个带黑色边框的色块
        patch = Patch(facecolor=color, edgecolor='black', label=label)
        # 如果是 'pd_' 类别，就给这个色块加上条纹
        if handle_idx in pd_indices:
            patch.set_hatch(hatch_pattern)
        new_handles.append(patch)
        
    fig.legend(
        new_handles, labels,
        loc='upper center',
        bbox_to_anchor=(0.5, 0.93),  # 定位到画布上方
        ncol=len(labels),            # 关键：水平排列列数等于类别数
        title='Match Type',
        frameon=False,
        title_fontsize=7,
        fontsize=6,
        columnspacing=0.8,          # 控制图例项间距
        handletextpad=0.3           # 控制符号与文本间距
    )

    # 调整整体布局
    plt.subplots_adjust(
        hspace=0.25,  # 垂直间距控制
        left=0.12,    # 左边界
        right=0.88,   # 右边界（需根据图例调整）
        bottom=0.08   # 底边界
    )
    plt.savefig(f'../../figures/apa_detect_{metric}_by_protocol.pdf', dpi=300, bbox_inches='tight')
    plt.show()  # 立即显示当前figure
    plt.close()  # 关闭当前figure释放内存
